# Group 18 — Debate-Driven Investment Committee
## Live multi-agent demonstration

This notebook runs the **existing project implementation**. It does not create mock agents or mock recommendations.

**Execution flow:** Research → Bull → Bear → Bull Rebuttal + Bear Rebuttal → Judge → Risk Agent → Final committee decision.

### Before running
1. Open this notebook from the root of `Debate-Driven-Investment-Committee`.
2. Activate the project's `.venv` and select that Python kernel in Jupyter/VS Code.
3. Confirm `.env` contains your Gemini API key (do not paste the key into this notebook).
4. Confirm the selected company has already been indexed in the project's ChromaDB database.

**Important:** The live execution calls Gemini and may consume API quota. Run the live committee cell once for your demonstration. It also appends a record to the project's run history.


In [ ]:
from pathlib import Path
import os
import sys
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()

# If the notebook was opened from a subfolder, locate the project root.
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "agents").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "agents").is_dir():
    raise RuntimeError(
        "Could not find the project root. Open this notebook from "
        "the Debate-Driven-Investment-Committee folder."
    )

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")

print("Project root:", PROJECT_ROOT)
print("Python executable:", sys.executable)
print("Gemini API key configured:", bool(os.getenv("GOOGLE_API_KEY")))


In [ ]:
# Check the local company indexes before starting a live run.
from tools.multi_company_rag import list_indexed_companies, get_company_document_count

companies = list_indexed_companies()

print("Indexed companies:")
for company in companies:
    try:
        count = get_company_document_count(company)
    except Exception:
        count = "count unavailable"
    print(f" • {company}: {count} indexed chunks")

if not companies:
    raise RuntimeError(
        "No indexed company collections were found. Upload and index a company "
        "document in the application before running this notebook."
    )


## Select the company

Set `COMPANY_NAME` to the exact company name shown in the indexed-company list above. The example below uses the first indexed company to avoid guessing a company name.


In [ ]:
COMPANY_NAME = companies[0]
print("Selected company:", COMPANY_NAME)


## Live committee execution

The next cell runs the real controller. It prints the agent execution trace and returns the final committee result. It may take several minutes depending on model response times.

Do not repeatedly execute this cell just to refresh the notebook: each execution starts a new committee run and appends another run-history record.


In [ ]:
import json
import subprocess
import sys

# Run the synchronous project controller in a separate Python process.
# This avoids nesting asyncio.run() inside Jupyter's already-running event loop.
runner = r"""
import json
from agents.committee_controller import run_committee

company = 'Jio Financial Services'
result = run_committee(company)
print("\\n=== NOTEBOOK_COMMITTEE_RESULT_JSON ===")
print(json.dumps(result, ensure_ascii=False, default=str))
"""

process = subprocess.run(
    [sys.executable, "-c", runner],
    cwd=str(PROJECT_ROOT),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
)

print(process.stdout)

if process.stderr.strip():
    print("\\n--- STDERR ---")
    print(process.stderr)

if process.returncode != 0:
    raise RuntimeError(
        f"Committee subprocess failed with exit code {process.returncode}. "
        "Read the output above; do not treat a fallback HOLD as a successful decision."
    )

marker = "=== NOTEBOOK_COMMITTEE_RESULT_JSON ==="
if marker not in process.stdout:
    raise RuntimeError(
        "The committee process did not return its result marker. "
        "The run may have failed before producing a final result."
    )

result_text = process.stdout.split(marker, 1)[1].strip()
result_line = result_text.splitlines()[0]
committee_result = json.loads(result_line)

print("\\n" + "=" * 65)
print("FINAL COMMITTEE RESULT")
print("=" * 65)
print("Company:", committee_result.get("company"))
print("Judge recommendation:", committee_result.get("judge_recommendation"))
print("Risk result:", committee_result.get("risk_result"))
print("Final recommendation:", committee_result.get("final_recommendation"))
print("Decision reason:", committee_result.get("decision_reason"))

if committee_result.get("risk_result") is None:
    raise RuntimeError(
        "The committee returned no Risk Agent result. "
        "Do not treat this execution as a completed committee decision."
    )


## Verify the debate and saved run

The controller prints the complete Research/Bull/Bear/Rebuttal/Judge/Risk execution trace above. The following cell checks that the latest run was appended to the JSONL run log.



In [ ]:
import json
from pathlib import Path

run_log = PROJECT_ROOT / "logs" / "committee_runs.jsonl"

if not run_log.exists():
    print("Run log not found:", run_log)
else:
    lines = [line for line in run_log.read_text(encoding="utf-8").splitlines() if line.strip()]
    print("Run log:", run_log)
    print("Total saved runs:", len(lines))
    latest = json.loads(lines[-1])
    print("Latest saved asset:", latest.get("asset", latest.get("company", "field not present")))
    print("Latest final recommendation:", latest.get("final_recommendation"))
    print("Latest decision reason:", latest.get("decision_reason"))


## Demonstration summary

Use the output above to explain the assignment requirements during the presentation:

- **Research Agent:** gathers facts from the selected company's indexed document collection.
- **Bull Agent:** argues the investment case using retrieved evidence.
- **Bear Agent:** argues the opposing case using retrieved evidence.
- **Rebuttal round:** Bull responds to Bear's objections and Bear responds to Bull's arguments.
- **Judge Agent:** evaluates the debate and issues BUY, HOLD, or AVOID with reasons.
- **Risk Agent:** calculates the configured screening score and can trigger the HOLD veto when the threshold is crossed.
- **RAG/shared memory:** FastEmbed creates local embeddings and ChromaDB stores/retrieves company-specific document chunks.
- **Run log:** the controller appends the execution record to `logs/committee_runs.jsonl`.

**Academic limitation:** The current Risk Agent uses a rule-based heuristic score. It is not a calibrated probability of loss or a credit rating. Evidence citations identify source filenames and internal chunk numbers, not PDF page numbers.
